# Validacao de IDs HardPos
Compara o mapeamento efetivo do treino com o usado na analise, sem modificar os dados.

In [1]:
import hashlib
import pickle
from pathlib import Path
import pandas as pd

root = Path('/home/mgteus/workspace/navitrans')
parquet = root / 'bert_trainset_squared_25.gzip'
expected_sha256 = 'ead5e3a708f06dad7b4982230d0de1db8ec4003356465049637ba13b06ec5723'

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

before = sha256(parquet)
assert before == expected_sha256, f'Parquet alterado antes da validacao: {before}'
df = pd.read_parquet(parquet)
with (root / 'ps_to_psid.pkl').open('rb') as handle:
    ps_to_psid = pickle.load(handle)
print('SHA-256 inicial:', before)
print('Linhas:', len(df), '| estados:', df.position_state.nunique())

SHA-256 inicial: ead5e3a708f06dad7b4982230d0de1db8ec4003356465049637ba13b06ec5723
Linhas: 13134 | estados: 947


In [2]:
estados = df[['position_state', 'position_state_id']].drop_duplicates().copy()
assert not estados['position_state'].duplicated().any(), 'Um estado possui varios IDs salvos'
estados['id_treino'] = estados['position_state'].map(ps_to_psid)
assert estados['id_treino'].notna().all(), 'Estado ausente no pickle'
assert not estados['id_treino'].duplicated().any(), 'ID de treino duplicado'
estados['id_analise'] = estados.groupby('position_state').ngroup()
divergentes = estados[estados['id_treino'] != estados['id_analise']]
print(f'Divergencias treino x analise: {len(divergentes)} de {len(estados)} estados')
print(f'Divergencias parquet original x treino: {(estados.position_state_id != estados.id_treino).sum()}')
print('Exemplos de divergencia:')
display(divergentes.head(12))
print('IDs ausentes no intervalo de treino:',
      sorted(set(range(int(estados.id_treino.min()), int(estados.id_treino.max()) + 1))
             - set(estados.id_treino.astype(int))))
after = sha256(parquet)
assert after == before, 'Parquet alterado durante a validacao'
print('SHA-256 final:', after)
print('divergencia treino x analise confirmada' if len(divergentes) else 'sem divergencia')

Divergencias treino x analise: 0 de 947 estados
Divergencias parquet original x treino: 945
Exemplos de divergencia:


,position_state,position_state_id,id_treino,id_analise


IDs ausentes no intervalo de treino: []
SHA-256 final: ead5e3a708f06dad7b4982230d0de1db8ec4003356465049637ba13b06ec5723
sem divergencia
